In [1]:
# Runtime -> Change runtime type -> T4 GPU
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

2.11.0+cu128
True
Tesla T4


# **PyTorch Core Components**

1. **Tensor (input data, weights, gradients)** (`torch.tensor`, `torch.zeros`, `torch.randn`) — the basic data structure; an n dimensional array that runs on CPU or GPU and can track gradients.

2. **Autograd** (`requires_grad=True`, `.backward()`, `.grad`) — automatic differentiation engine that tracks operations and computes gradients.

3. **nn.Module** (`class Net(nn.Module)`, `forward`) — base class for models and layers; subclass it to define a network.

4. **Layers** (`nn.Linear`, `nn.Conv2d`, `nn.ReLU`, `nn.Dropout`, `nn.Sequential`) — the building blocks of a network.

5. **Loss function** (`nn.CrossEntropyLoss`, `nn.MSELoss`, `nn.BCEWithLogitsLoss`) — measures prediction error.

6. **Optimizer** (`torch.optim.SGD`, `torch.optim.Adam`) — updates weights from gradients.

7. **Dataset and DataLoader** (`torch.utils.data.Dataset`, `DataLoader`) — hold samples and provide batching, shuffling, and iteration.

8. **Training loop** (`optimizer.zero_grad()`, `model(inputs)`, `loss.backward()`, `optimizer.step()`) — the repeated training cycle.

9. **Device** (`torch.device("cuda")`, `.to(device)`) — `cpu` or `cuda`; tensors and models must share a device.

10. **state_dict** (`torch.save`, `model.load_state_dict`, `torch.load`) — learned weights, used for saving and loading.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np

In [3]:
X, y = load_breast_cancer(return_X_y=True)
print(X.shape, y.shape)
feature_names = load_breast_cancer().feature_names

(569, 30) (569,)


In [4]:
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
sample_df = pd.concat([
    df[df['target'] == 0].head(3),
    df[df['target'] == 1].head(3)
])
sample_df.head(10)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.990,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.71190,0.26540,0.4601,0.11890,0
1,20.570,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.24160,0.18600,0.2750,0.08902,0
2,19.690,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.45040,0.24300,0.3613,0.08758,0
19,13.540,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.04781,0.1885,0.05766,...,19.26,99.70,711.2,0.1440,0.1773,0.23900,0.12880,0.2977,0.07259,1
20,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.03110,0.1967,0.06811,...,20.49,96.09,630.5,0.1312,0.2776,0.18900,0.07283,0.3184,0.08183,1
21,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.02076,0.1815,0.06905,...,15.66,65.13,314.9,0.1324,0.1148,0.08867,0.06227,0.2450,0.07773,1


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler() # Z-score normalization
X_train = scaler.fit_transform(X_train) # calculates mean and sigma
X_test = scaler.transform(X_test) # uses mean and sigma from X_train

In [6]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

In [7]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [8]:
# NNs are implemented as classes since there are so many things to customize
class BCNet(nn.Module):
  def __init__(self):
    super(BCNet, self).__init__()

    # Funnel structure start wide to capture complex feature combinations & gradually compress that information down toward your final prediction
    self.fc1 = nn.Linear(30, 64)
    self.fc2 = nn.Linear(64, 32)
    self.fc3 = nn.Linear(32, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = torch.sigmoid(self.fc3(x))
    return x

In [9]:
criterion = nn.BCELoss()
model = BCNet()
#optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

In [10]:
epochs = 20
error = []

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for x_batch, y_batch in train_loader:
    optimizer.zero_grad()

    preds = model(x_batch)
    loss = criterion(preds, y_batch)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f"Epoch {epoch+1}: Loss {running_loss/len(train_loader)}")
  error.append(running_loss/len(train_loader))

Epoch 1: Loss 0.6675002773602804
Epoch 2: Loss 0.5238679150740305
Epoch 3: Loss 0.33458700676759084
Epoch 4: Loss 0.19689940909544626
Epoch 5: Loss 0.13838875343402227
Epoch 6: Loss 0.1117242102821668
Epoch 7: Loss 0.09271378889679908
Epoch 8: Loss 0.08044729880057275
Epoch 9: Loss 0.0747948714842399
Epoch 10: Loss 0.07328593668838342
Epoch 11: Loss 0.064191233428816
Epoch 12: Loss 0.061282623559236526
Epoch 13: Loss 0.05699436195815603
Epoch 14: Loss 0.0597108731046319
Epoch 15: Loss 0.052072636193285386
Epoch 16: Loss 0.050215971066306035
Epoch 17: Loss 0.04887724332511425
Epoch 18: Loss 0.04621000123831133
Epoch 19: Loss 0.044964757810036345
Epoch 20: Loss 0.043541506056984265


In [11]:
model.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation for evaluation
    y_pred_tensor = model(X_test_tensor)
    y_pred_binary = (y_pred_tensor >= 0.5).float() # Convert probabilities to binary (0 or 1)

accuracy = accuracy_score(y_test_tensor.numpy(), y_pred_binary.numpy())
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9737


In [16]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42, C=0.8)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Accuracy: {accuracy:.4f}")


Logistic Regression Accuracy: 0.9737
